# IMUSA: Multimodal Punjabi Meme Sentiment Analysis — Training & Inference Pipeline

This notebook trains a Late-Fusion Multimodal Model (**Vision Transformer + XLM-RoBERTa + Gated Fusion + Focal Loss**) for 4-class sentiment classification on Punjabi memes (`Sarcasm`, `Motivational`, `Neutral`, `Offensive`).

**Hardware**: Google Colab T4 GPU (Free tier)

## 1. Environment Setup & Repository Clone

In [ ]:
!pip install -q uv
!git clone https://github.com/shubhojit-mitra-dev/imusa-multimodal-sentiment.git project
%cd project
!uv sync --all-packages

## 2. Dataset Setup (Upload / Unzip data.zip)

Upload your `data.zip` file containing the `data/` directory.

In [ ]:
from pathlib import Path

train_csv = Path("data/train/train_punjabi_dataset.csv")

if not train_csv.exists():
    print("Dataset not detected in project/data/")
    if Path("/content/data.zip").exists():
        print("Found /content/data.zip, extracting...")
        !unzip -q /content/data.zip -d /content/project/
    elif Path("data.zip").exists():
        print("Found data.zip in project directory, extracting...")
        !unzip -q data.zip -d /content/project/
    else:
        print("Please upload your 'data.zip' file:")
        from google.colab import files

        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith(".zip"):
                !unzip -q "{fname}" -d /content/project/

if train_csv.exists():
    print("✅ Dataset successfully verified at data/train/train_punjabi_dataset.csv")
else:
    print("❌ Dataset missing. Please ensure data.zip contains data/ directory structure.")

## 3. Execute Dataset Cleaning & EDA Pipeline

In [ ]:
!uv run python scripts/clean_data.py
!uv run python scripts/explore_data.py

## 4. Train Multimodal Model with Cosine Warmup & Focal Loss

In [ ]:
!uv run python scripts/train.py --epochs 10 --batch-size 16 --lr 2e-5 --loss focal --warmup-ratio 0.1

## 5. Run Test Set Inference & Generate Submission CSV

In [ ]:
!uv run python scripts/predict.py --checkpoint outputs/checkpoints/best_model.pt --output outputs/submission.csv

## 6. Download Results & Submission Artifacts

In [ ]:
from google.colab import files

files.download("outputs/submission.csv")
files.download("outputs/checkpoints/best_model.pt")